# Entrenamiento con LANDMARKS y Videos Courtesy

Este notebook crea una intersección controlada entre el dataset original de landmarks y Videos Courtesy. Conserva solo 13 coincidencias exactas. Añade cuatro rótulos Courtesy semánticamente relacionados como **clases aparte**, no fusionadas con la seña original: `0050` Como esta, `0051` Chao, `0052` Denada y `0053` Lo siento.

El preprocesamiento extrae pose y manos con MediaPipe a CSV `clase_participante_toma.csv`, prepara los landmarks, combina fuentes con IDs de firmante aislados, entrena el Transformer y presenta curvas, distribución por participante, matriz de confusión y métricas por clase. La evaluación reserva firmantes completos. La extracción de los AVI puede tardar.

El ZIP adjunto contiene 1.228 AVI y el inventario inicial encontró 737 videos de las clases seleccionadas. La celda de manifiesto verifica el número final de extracciones correctas.

Este flujo es independiente de SPOTER; no modifica sus archivos.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive')
    MYDRIVE = Path('/content/drive/MyDrive')
    project_candidates = [
        MYDRIVE / 'FASE 2 IA' / 'Fase 2' / 'Codigo' / 'landmark_recognition',
        MYDRIVE / 'Codigo' / 'landmark_recognition',
        MYDRIVE / 'landmark_recognition',
    ]
    PROJECT_DIR = next((path for path in project_candidates if path.is_dir()), None)
    if PROJECT_DIR is None:
        raise FileNotFoundError('Copia landmark_recognition completa a Google Drive antes de ejecutar.')
    search_roots = [MYDRIVE, MYDRIVE / 'FASE 2 IA' / 'Fase 2', PROJECT_DIR]
    ARCHIVE_PATH = next(
        (root / 'Videos Courtesy.zip' for root in search_roots if (root / 'Videos Courtesy.zip').is_file()),
        None,
    )
    if ARCHIVE_PATH is None:
        raise FileNotFoundError('Sube Videos Courtesy.zip a Mi unidad, FASE 2 IA/Fase 2 o landmark_recognition.')
    LANDMARKS_ARCHIVE_PATH = next(
        (root / 'LANDMARKS.zip' for root in search_roots if (root / 'LANDMARKS.zip').is_file()),
        None,
    )
except ModuleNotFoundError:
    IN_COLAB = False
    cwd = Path.cwd().resolve()
    project_candidates = [
        cwd if (cwd / 'courtesy_preprocessing').is_dir() else cwd / 'landmark_recognition',
        cwd / 'Codigo' / 'landmark_recognition',
        cwd.parent / 'Codigo' / 'landmark_recognition',
    ]
    PROJECT_DIR = next((path for path in project_candidates if path.is_dir()), None)
    if PROJECT_DIR is None:
        raise FileNotFoundError('Abre este notebook desde Fase 2, Codigo o landmark_recognition.')
    search_roots = [PROJECT_DIR, PROJECT_DIR.parent, PROJECT_DIR.parent.parent, PROJECT_DIR.parent.parent.parent, cwd]
    ARCHIVE_PATH = next(
        (root / 'Videos Courtesy.zip' for root in search_roots if (root / 'Videos Courtesy.zip').is_file()),
        None,
    )
    if ARCHIVE_PATH is None:
        raise FileNotFoundError('No se encontr? Videos Courtesy.zip junto al proyecto ni en sus carpetas superiores.')
    LANDMARKS_ARCHIVE_PATH = next(
        (root / 'LANDMARKS.zip' for root in search_roots if (root / 'LANDMARKS.zip').is_file()),
        None,
    )

sys.path.insert(0, str(PROJECT_DIR.parent))
PREPROCESSING_DIR = PROJECT_DIR / 'courtesy_preprocessing'
OUTPUT_DIR = PREPROCESSING_DIR / 'outputs'
CSV_DIR = OUTPUT_DIR / 'landmarks_csv'
COURTESY_DATASET_PATH = PROJECT_DIR / 'data' / 'courtesy_videos.npz'
BASE_DATASET_PATH = PROJECT_DIR / 'data' / 'landmarks_dataset.npz'
COMBINED_DATASET_PATH = PROJECT_DIR / 'data' / 'landmarks_intersection.npz'
CHECKPOINT_PATH = PROJECT_DIR / 'checkpoints' / 'transformer_intersection.pth'
RESULTS_DIR = PROJECT_DIR / 'reports' / 'courtesy_intersection'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('Proyecto:', PROJECT_DIR)
print('ZIP de videos:', ARCHIVE_PATH, f'({ARCHIVE_PATH.stat().st_size / 1e6:.1f} MB)')

## 1. Reglas de correspondencia

El código de clase original se conserva solo para coincidencias exactas tras normalizar mayúsculas, tildes y espacios. Las cuatro equivalencias semánticas reciben códigos nuevos (`0050`–`0053`) para que el modelo pueda diferenciarlas.

In [ ]:
from landmark_recognition.courtesy_preprocessing.class_mapping import (
    DIRECT_VIDEO_LABELS, SEMANTIC_VIDEO_LABELS, DIRECT_CLASSES,
    SEMANTIC_CLASSES, COMBINED_CLASSES, map_video_label,
)
from landmark_recognition.class_labels import CLASS_LABELS

mapping_rows = []
for label, code in DIRECT_VIDEO_LABELS.items():
    mapping_rows.append({'Tipo': 'Coincidencia exacta', 'Etiqueta Courtesy': label, 'Código final': code, 'Nombre': CLASS_LABELS[code]})
for label, code in SEMANTIC_VIDEO_LABELS.items():
    mapping_rows.append({'Tipo': 'Clase semántica independiente', 'Etiqueta Courtesy': label, 'Código final': code, 'Nombre': CLASS_LABELS[code]})
display(pd.DataFrame(mapping_rows).sort_values('Código final').reset_index(drop=True))
assert len(DIRECT_CLASSES) == 13 and len(SEMANTIC_CLASSES) == 4 and len(COMBINED_CLASSES) == 17

## 2. Extraer landmarks de los AVI

El extractor selecciona únicamente las carpetas etiquetadas en el mapeo, procesa hasta 96 frames uniformemente por video y escribe el CSV y el manifiesto en una carpeta de preprocesamiento separada. Se puede reanudar: los CSV existentes se omiten. Usa `overwrite=True` si quieres rehacerlos.

In [ ]:
from landmark_recognition.courtesy_preprocessing.extract_video_landmarks import extract_archive

MANIFEST_PATH = extract_archive(
    archive_path=ARCHIVE_PATH,
    output_dir=OUTPUT_DIR,
    max_frames=96,
    overwrite=False,
)
manifest = pd.read_csv(MANIFEST_PATH)
manifest['status'].value_counts()

In [ ]:
failures = manifest[manifest['status'] == 'error']
if not failures.empty:
    display(failures[['video_member', 'error']].head(20))
    raise RuntimeError(f'{len(failures)} videos fallaron; revisa {MANIFEST_PATH}')

successful = manifest[manifest['status'].isin(['ok', 'skipped_existing'])]
print('Videos seleccionados:', len(manifest))
print('Landmarks disponibles:', len(successful))
print('Códigos presentes:', sorted(manifest['class_name'].unique()))
fig, axes = plt.subplots(1, 2, figsize=(17, 5))
manifest.groupby('class_name').size().reindex(COMBINED_CLASSES, fill_value=0).plot.bar(ax=axes[0], title='Videos por clase')
manifest.groupby('signer').size().sort_index().plot.bar(ax=axes[1], title='Videos seleccionados por participante')
axes[0].set_xlabel('Código'); axes[1].set_xlabel('Participante')
plt.tight_layout(); fig.savefig(RESULTS_DIR / 'extraction_distribution.png', dpi=150); plt.show()

## 3. Preparar el dataset de videos y combinarlo

El dataset Courtesy se convierte a 64 frames. Después se filtra el dataset original a las 13 coincidencias exactas y se agregan las cuatro clases Courtesy aparte. Los IDs de firmante llevan prefijo de fuente durante la combinación para evitar que participantes distintos con el mismo número se confundan.

In [ ]:
from landmark_recognition.prepare_landmark_dataset import prepare_dataset
from landmark_recognition.merge_landmark_datasets import merge_datasets

if not COURTESY_DATASET_PATH.is_file():
    prepare_dataset(CSV_DIR, COURTESY_DATASET_PATH, frame_count=64)
else:
    print('Dataset Courtesy existente:', COURTESY_DATASET_PATH)

if not BASE_DATASET_PATH.is_file():
    if LANDMARKS_ARCHIVE_PATH is None or not LANDMARKS_ARCHIVE_PATH.is_file():
        raise FileNotFoundError(
            f'Falta {BASE_DATASET_PATH} y no se encontró LANDMARKS.zip. '
            'Coloca el NPZ preparado o el ZIP original junto a los datos.'
        )
    prepare_dataset(LANDMARKS_ARCHIVE_PATH, BASE_DATASET_PATH, frame_count=64)

merge_datasets(
    [BASE_DATASET_PATH, COURTESY_DATASET_PATH],
    COMBINED_DATASET_PATH,
    keep_classes=list(COMBINED_CLASSES),
)
with np.load(COMBINED_DATASET_PATH, allow_pickle=False) as data:
    X = data['X'].astype(np.float32)
    y = data['y'].astype(np.int64)
    signers = data['signers'].astype(str)
    sample_ids = data['sample_ids'].astype(str)
    class_names = data['class_names'].astype(str)
assert set(class_names) == set(COMBINED_CLASSES)
print('Secuencias:', X.shape, '| clases:', len(class_names), '| firmantes:', len(np.unique(signers)))

In [ ]:
source_names = np.asarray([sample_id.split('/', 1)[0] for sample_id in sample_ids])
class_index = {name: index for index, name in enumerate(class_names)}
class_counts = pd.DataFrame({
    'Código': class_names,
    'Seña': [CLASS_LABELS[name] for name in class_names],
    'LANDMARKS': [int(np.sum((y == class_index[name]) & (source_names == BASE_DATASET_PATH.stem))) for name in class_names],
    'Courtesy': [int(np.sum((y == class_index[name]) & (source_names == COURTESY_DATASET_PATH.stem))) for name in class_names],
})
display(class_counts)
class_counts.to_csv(RESULTS_DIR / 'class_counts.csv', index=False)
fig, ax = plt.subplots(figsize=(16, 5))
class_counts.set_index('Código')[['LANDMARKS', 'Courtesy']].plot.bar(ax=ax, stacked=True, color=['#3973ac', '#e29b32'])
ax.set_title('Ejemplos por clase y fuente'); ax.set_ylabel('Secuencias'); ax.set_xlabel('Código de clase')
plt.tight_layout(); fig.savefig(RESULTS_DIR / 'class_counts.png', dpi=150); plt.show()

coverage = pd.crosstab(signers, y).reindex(columns=np.arange(len(class_names)), fill_value=0)
print('Firmantes con cobertura de las 17 clases:', int((coverage > 0).all(axis=1).sum()))
display(coverage.iloc[:10])

## 4. Entrenar con separación por participante

El notebook elige dos participantes de Courtesy que tengan ejemplos de las 17 clases: uno para validación y otro para prueba. Ningún video de esos participantes interviene en el ajuste de parámetros. Si no hay al menos tres participantes con las 17 clases, se detiene en vez de presentar una evaluación incompleta como si cubriera todo el vocabulario.

In [ ]:
from types import SimpleNamespace
from landmark_recognition.train_model import train

courtesy_signers = sorted({signer for signer, sample_id in zip(signers, sample_ids) if sample_id.startswith(COURTESY_DATASET_PATH.stem + '/')})
coverage = pd.crosstab(signers, y).reindex(columns=np.arange(len(class_names)), fill_value=0)
full_coverage_signers = [signer for signer in courtesy_signers if (coverage.loc[signer] > 0).all()]
if len(full_coverage_signers) < 3:
    raise ValueError(
        f'Solo {len(full_coverage_signers)} participantes Courtesy tienen las 17 clases. '
        'Agrega más participantes completos para reservar validación y prueba sin clases ausentes.'
    )
validation_signer, test_signer = full_coverage_signers[-2:]
print('Firmante de validación:', validation_signer)
print('Firmante de prueba:', test_signer)

args = SimpleNamespace(
    dataset=COMBINED_DATASET_PATH,
    architecture='transformer',
    output=CHECKPOINT_PATH,
    epochs=100,
    batch_size=16,
    learning_rate=0.0005,
    validation_signer=validation_signer,
    test_signer=test_signer,
    split_mode='subject',
    no_augmentation=False,
    signer_invariant=True,
    wrist_relative_hands=False,
    include_velocity=False,
    no_mean_shift=False,
    pooling='mean',
    seed=42,
    early_stopping_patience=20,
    early_stopping_min_delta=0.001,
)
train(args)

## 5. Curvas de aprendizaje

In [ ]:
import torch

checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=True)
history = checkpoint['history']
epochs = np.arange(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(epochs, history['train_loss'], label='Entrenamiento')
axes[0].plot(epochs, history['validation_loss'], label='Validación')
axes[0].set_title('Pérdida'); axes[0].set_xlabel('Época'); axes[0].legend(); axes[0].grid(alpha=.25)
axes[1].plot(epochs, history['train_accuracy'], label='Entrenamiento')
axes[1].plot(epochs, history['validation_accuracy'], label='Validación')
axes[1].set_title('Exactitud'); axes[1].set_xlabel('Época'); axes[1].legend(); axes[1].grid(alpha=.25)
plt.tight_layout(); fig.savefig(RESULTS_DIR / 'training_curves.png', dpi=150); plt.show()
print('Mejor época:', checkpoint['best_epoch'])
print('Exactitud validación:', checkpoint['validation_accuracy'])
print('Exactitud prueba:', checkpoint['test_accuracy'])

## 6. Matriz de confusión y métricas por seña

Las predicciones guardadas corresponden únicamente al participante reservado para prueba. Se muestran conteos reales/predichos y exactitud por clase; las celdas del conjunto de prueba tienen tres videos Courtesy por seña para el firmante seleccionado.

In [ ]:
targets = np.asarray(checkpoint['test_targets'], dtype=int)
predictions = np.asarray(checkpoint['test_predictions'], dtype=int)
if not len(targets):
    raise ValueError('El checkpoint no contiene predicciones de prueba.')
confusion = np.zeros((len(class_names), len(class_names)), dtype=int)
np.add.at(confusion, (targets, predictions), 1)
per_class = []
for index, code in enumerate(class_names):
    total = int(confusion[index].sum())
    per_class.append({'Código': code, 'Seña': CLASS_LABELS[code], 'Correctas': int(confusion[index, index]), 'Prueba': total, 'Exactitud': confusion[index, index] / total if total else np.nan})
per_class_df = pd.DataFrame(per_class)
display(per_class_df)
per_class_df.to_csv(RESULTS_DIR / 'test_metrics_per_class.csv', index=False)
fig, ax = plt.subplots(figsize=(13, 11))
image = ax.imshow(confusion, cmap='Blues')
ax.set_title('Matriz de confusión — participante reservado')
ax.set_xlabel('Predicción'); ax.set_ylabel('Etiqueta real')
ax.set_xticks(np.arange(len(class_names)), class_names, rotation=90)
ax.set_yticks(np.arange(len(class_names)), class_names)
fig.colorbar(image, ax=ax, label='Videos')
plt.tight_layout(); fig.savefig(RESULTS_DIR / 'confusion_matrix.png', dpi=150); plt.show()
print(f'Exactitud del participante de prueba: {(targets == predictions).mean():.3f}')

## Inferencia con webcam

Descarga `transformer_intersection.pth` a la computadora conectada a la cámara y ejecuta desde `Codigo`:

```powershell
python -m landmark_recognition.predict_video `
  --checkpoint landmark_recognition\checkpoints\transformer_intersection.pth `
  --source 0
```

Para Colab, actualiza la ruta del checkpoint en la celda de `run_colab_webcam` del notebook de análisis.